# Bài 8: AI tạo sinh nhập môn
**Mở hộp đen AI · Tập 1 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Các câu văn trong bài do nhóm tác giả tự viết về một lớp học tưởng tượng.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 1. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Kho chữ: 34 câu về lớp 10A3

In [ ]:
kho = [
    "cô hạnh dạy toán ở lớp mười a ba", "thầy nam dạy lý ở lớp mười a ba",
    "cô mai dạy văn ở lớp mười a ba", "cô hạnh rất hiền", "thầy nam rất vui tính",
    "cô mai rất nghiêm", "minh thích học toán", "lan thích học văn",
    "khoa thích học lý", "minh thích chơi bóng đá", "lan thích đọc sách",
    "khoa thích chơi cờ vua", "em thích học toán với cô hạnh",
    "em thích học văn với cô mai", "em thích học lý với thầy nam",
    "sáng thứ hai lớp em chào cờ", "chiều thứ năm lớp em đá bóng",
    "em đi học bằng xe đạp", "em đi học bằng xe buýt", "minh đi học bằng xe đạp",
    "lan đi học bằng xe buýt", "khoa đi học bằng xe máy điện",
    "lớp em có ba mươi lăm bạn", "lớp em học ở tầng hai",
    "thư viện trường em ở tầng một", "em đọc sách ở thư viện",
    "em ăn sáng ở căng tin", "căng tin trường em bán bánh mì",
    "mai lớp em kiểm tra toán", "thứ sáu lớp em kiểm tra văn",
    "cô hạnh chấm bài rất nhanh", "thầy nam cho nhiều bài tập",
    "cô mai kể chuyện rất hay", "em rất thích lớp mười a ba",
]
print("Số câu:", len(kho))

### Ô 2. Đếm xem sau mỗi từ thường là từ gì

In [ ]:
from collections import Counter, defaultdict

tu_tiep = defaultdict(Counter)        # tu_tiep["em"]["thích"] = số lần "em thích"
for cau in kho:
    tu = ["<đầu>"] + cau.split() + ["<hết>"]
    for a, b in zip(tu, tu[1:]):
        tu_tiep[a][b] += 1

print("Số từ khác nhau:", len(tu_tiep))
print('Sau "em" là:', tu_tiep["em"].most_common(5))

### Ô 3. Biến số đếm thành xác suất

In [ ]:
def xac_suat(tu_truoc):
    dem = tu_tiep[tu_truoc]
    tong = sum(dem.values())
    return {b: so / tong for b, so in dem.most_common()}

for b, p in xac_suat("em").items():
    print(f'P({b} | em) = {p:.2f}')

### Ô 4. Cho máy tự viết câu
Mỗi bước, máy bốc thăm từ tiếp theo theo xác suất, đến khi gặp `<hết>`.

In [ ]:
import random

def sinh_cau(nhiet_do=1.0, toi_da=15):
    tu, cau = "<đầu>", []
    while len(cau) < toi_da:
        ung_vien = list(tu_tiep[tu].keys())
        trong_so = [so ** (1 / nhiet_do) for so in tu_tiep[tu].values()]
        tu = random.choices(ung_vien, weights=trong_so)[0]
        if tu == "<hết>":
            break
        cau.append(tu)
    return " ".join(cau)

random.seed(3)
for _ in range(6):
    c = sinh_cau()
    print(("[có sẵn] " if c in kho else "[câu mới] ") + c)

## Chặng 2: Đọc
Trả lời vào vở trước khi đi tiếp:
1. Ở ô 2, máy lưu lại điều gì? Nó có lưu nghĩa của từ không?
2. Vì sao máy viết được những câu không hề có trong kho chữ?
3. Đọc các câu mới ở ô 4. Câu nào **đúng ngữ pháp nhưng sai sự thật**, nếu so với kho chữ?

## Chặng 3: Sửa
### Ô 5. Vặn 'nhiệt độ': thấp thì an toàn, cao thì liều

In [ ]:
for t in [0.3, 1.0, 3.0]:
    random.seed(0)
    print(f"--- nhiệt độ {t} ---")
    for _ in range(3):
        print("  " + sinh_cau(nhiet_do=t))

### Ô 6. Đếm 'ảo giác': máy viết sai sự thật bao nhiêu lần?
Trong kho chữ: cô Hạnh dạy toán, thầy Nam dạy lý, cô Mai dạy văn. Ta cho máy viết 1.000 câu rồi kiểm tra những câu nói ai dạy môn gì.

In [ ]:
SU_THAT = {"cô hạnh": "toán", "thầy nam": "lý", "cô mai": "văn"}

random.seed(1)
cau_ra = [sinh_cau() for _ in range(1000)]
noi_ve_day = [c for c in cau_ra if " dạy " in c]
sai = []
for c in noi_ve_day:
    for gv, mon in SU_THAT.items():
        if c.startswith(gv + " dạy ") and not c.startswith(gv + " dạy " + mon):
            sai.append(c)

print("Số câu có chữ 'dạy':", len(noi_ve_day))
print("Số câu nói SAI ai dạy môn gì:", len(sai))
for c in sorted(set(sai))[:3]:
    print("  ví dụ:", c)

### Ô 7. Em mở đầu, máy viết tiếp

In [ ]:
def noi_tiep(mo_dau, nhiet_do=1.0):
    cau = mo_dau.lower().split()
    tu = cau[-1]
    while len(cau) < 15 and tu in tu_tiep:
        tu = random.choices(list(tu_tiep[tu].keys()),
                            weights=[so ** (1 / nhiet_do) for so in tu_tiep[tu].values()])[0]
        if tu == "<hết>":
            break
        cau.append(tu)
    return " ".join(cau)

print(noi_tiep(input("Gõ vài từ mở đầu: ")))

## Chặng 4: Tự làm
1. Thay kho chữ bằng 20 câu ca dao, tục ngữ mà em thuộc rồi chạy lại ô 2 đến ô 4. Máy 'sáng tác' ra những câu như thế nào?
2. Viết hàm `xac_suat_cau(cau)` tính xác suất của cả một câu bằng cách nhân các xác suất từng bước. Câu nào trong kho chữ có xác suất cao nhất?
3. Sửa ô 6 để kiểm tra thêm một loại sự thật khác: ai đi học bằng phương tiện gì.

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*